In [1]:
# Import libraries 
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)  

In [2]:
# Load processed dataset 
df = pd.read_csv(
    "../data/processed/processed_data.csv"
)

print("Dataset Shape:", df.shape)

df.head() 

Dataset Shape: (1500, 19)


,age,gender,temperature,heart_rate,blood_pressure,glucose,cholesterol,fever,cough,fatigue,headache,chest_pain,breathing_difficulty,nausea,disease,symptom_count,age_group,blood_pressure_category,glucose_category
0,65.0,Female,98.3,71,176.0,130.0,236,Yes,No,No,NaN,Yes,Yes,Yes,Pneumonia,4.0,Senior,High,High
1,19.0,Female,97.6,72,124.0,115.0,132,No,No,No,Yes,Yes,Yes,No,Migraine,3.0,Young_Adult,Elevated,Prediabetic
2,NaN,Female,98.9,96,145.0,154.0,244,Yes,Yes,No,No,No,Yes,Yes,Hypertension,4.0,NaN,High,High
3,45.0,Male,99.5,99,168.0,172.0,275,Yes,Yes,Yes,No,No,No,Yes,Diabetes,4.0,Middle_Aged,High,High
4,70.0,Male,100.6,71,142.0,NaN,181,Yes,Yes,No,No,Yes,No,Yes,Flu,4.0,Senior,High,NaN


In [3]:
# Check columns 
print("Dataset Columns:") 
print(df.columns.tolist()) 

Dataset Columns:
['age', 'gender', 'temperature', 'heart_rate', 'blood_pressure', 'glucose', 'cholesterol', 'fever', 'cough', 'fatigue', 'headache', 'chest_pain', 'breathing_difficulty', 'nausea', 'disease', 'symptom_count', 'age_group', 'blood_pressure_category', 'glucose_category']


In [4]:
# Separate features and target 
X = df.drop(
    "disease",
    axis=1
) 
y = df["disease"] 
print("X Shape:", X.shape)
print("y Shape:", y.shape) 
print("\nTarget Classes:")
print(y.unique()) 

X Shape: (1500, 18)
y Shape: (1500,)

Target Classes:
['Pneumonia' 'Migraine' 'Hypertension' 'Diabetes' 'Flu']


In [5]:
# Define numerical features  
numerical_features = [
    "age",
    "temperature",
    "heart_rate",
    "blood_pressure",
    "glucose",
    "cholesterol",
    "symptom_count"
]

print("Numerical Features:")

print(numerical_features) 

Numerical Features:
['age', 'temperature', 'heart_rate', 'blood_pressure', 'glucose', 'cholesterol', 'symptom_count']


In [6]:
# Define categorical features 
categorical_features = [
    "gender",
    "fever",
    "cough",
    "fatigue",
    "headache",
    "chest_pain",
    "breathing_difficulty",
    "nausea",
    "age_group",
    "blood_pressure_category",
    "glucose_category"
]

print("Categorical Features:")

print(categorical_features) 

Categorical Features:
['gender', 'fever', 'cough', 'fatigue', 'headache', 'chest_pain', 'breathing_difficulty', 'nausea', 'age_group', 'blood_pressure_category', 'glucose_category']


In [7]:
# Numerical preprocessing 
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
) 

In [8]:
# Categorical preprocessing 
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
) 

In [9]:
# Create ColumnTransformer 
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_pipeline,
            numerical_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)

print("Preprocessor created successfully.") 

Preprocessor created successfully.


In [10]:
# Create Random Forest pipeline 
pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            RandomForestClassifier(
                random_state=42,
                class_weight="balanced"
            )
        )
    ]
)

pipeline 

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [11]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Data:", X_train.shape)
print("Testing Data :", X_test.shape) 

Training Data: (1200, 18)
Testing Data : (300, 18)


In [12]:
# Define hyperparameter grid 
param_grid = {
    "model__n_estimators": [
        100,
        200
    ],

    "model__max_depth": [
        None,
        10,
        20
    ],

    "model__min_samples_split": [
        2,
        5
    ],

    "model__min_samples_leaf": [
        1,
        2
    ]
}

param_grid 

{'model__n_estimators': [100, 200],
 'model__max_depth': [None, 10, 20],
 'model__min_samples_split': [2, 5],
 'model__min_samples_leaf': [1, 2]}

In [13]:
# Create GridSearchCV  
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)

print("GridSearchCV created successfully.") 

GridSearchCV created successfully.


In [14]:
# Train / Hyperparameter tuning 
print("Starting hyperparameter tuning...")

grid_search.fit(
    X_train,
    y_train
)

print("Hyperparameter tuning completed.") 

Starting hyperparameter tuning...
Fitting 5 folds for each of 24 candidates, totalling 120 fits
Hyperparameter tuning completed.


In [15]:
# Best parameters 
print("Best Parameters:")

print(
    grid_search.best_params_
) 

Best Parameters:
{'model__max_depth': 10, 'model__min_samples_leaf': 1, 'model__min_samples_split': 5, 'model__n_estimators': 100}


In [16]:
# Best CV score 
print("Best Cross-Validation F1 Score:")

print(
    grid_search.best_score_
) 

Best Cross-Validation F1 Score:
0.356371778041037


In [17]:
# Get best model 
best_model = grid_search.best_estimator_

print("Best model:")
print(best_model) 

Best model:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'temperature',
                                                   'heart_rate',
                                                   'blood_pressure', 'glucose',
                                                   'cholesterol',
                                                   'symptom_count']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                

In [18]:
# Test prediction 
predictions = best_model.predict(
    X_test
)

print("Predictions generated successfully.")

print(predictions[:20]) 

Predictions generated successfully.
['Migraine' 'Flu' 'Diabetes' 'Pneumonia' 'Flu' 'Migraine' 'Flu'
 'Pneumonia' 'Flu' 'Migraine' 'Hypertension' 'Diabetes' 'Flu' 'Pneumonia'
 'Pneumonia' 'Hypertension' 'Diabetes' 'Flu' 'Hypertension' 'Diabetes']


In [19]:
# Accuracy 
accuracy = accuracy_score(
    y_test,
    predictions
)

print(
    f"Accuracy: {accuracy:.4f}"
) 

Accuracy: 0.3433


In [20]:
# Precision 
precision = precision_score(
    y_test,
    predictions,
    average="macro",
    zero_division=0
)

print(
    f"Precision: {precision:.4f}"
) 

Precision: 0.3380


In [21]:
# Recall 
recall = recall_score(
    y_test,
    predictions,
    average="macro",
    zero_division=0
)

print(
    f"Recall: {recall:.4f}"
) 

Recall: 0.3377


In [22]:
# F1 Score 
f1 = f1_score(
    y_test,
    predictions,
    average="macro",
    zero_division=0
)

print(
    f"F1 Score: {f1:.4f}"
)

F1 Score: 0.3310


In [23]:
# Final metrics together 
print("=" * 60)
print("FINAL MODEL PERFORMANCE")
print("=" * 60)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}") 

FINAL MODEL PERFORMANCE
Accuracy : 0.3433
Precision: 0.3380
Recall   : 0.3377
F1 Score : 0.3310


In [24]:
# Classification report 
print("Classification Report:")

print(
    classification_report(
        y_test,
        predictions,
        zero_division=0
    )
)

Classification Report:
              precision    recall  f1-score   support

    Diabetes       0.36      0.29      0.32        56
         Flu       0.36      0.38      0.37        69
Hypertension       0.34      0.53      0.42        60
    Migraine       0.40      0.32      0.36        62
   Pneumonia       0.23      0.17      0.19        53

    accuracy                           0.34       300
   macro avg       0.34      0.34      0.33       300
weighted avg       0.34      0.34      0.34       300



In [25]:
# Save best model 
os.makedirs(
    "../models",
    exist_ok=True
)

model_path = "../models/best_model.pkl"

joblib.dump(
    best_model,
    model_path
)

print("Best model saved successfully.")

print(model_path)

Best model saved successfully.
../models/best_model.pkl


In [26]:
# Verify saved model 
loaded_model = joblib.load(
    "../models/best_model.pkl"
)

print("Model loaded successfully.")

print(
    type(loaded_model)
) 

Model loaded successfully.
<class 'sklearn.pipeline.Pipeline'>
